In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp
from scipy.stats import chi2_contingency

In [2]:
df = pd.read_parquet("../data_raw_clean/models.parquet")[["fullname", "Architecture", "#Params (B)", "cohort"]]
# df = df[df["Architecture"] == "LlamaForCausalLM"]

In [6]:
df["Architecture"].value_counts()

Architecture
LlamaForCausalLM                   1849
Qwen2ForCausalLM                   1308
MistralForCausalLM                  615
Gemma2ForCausalLM                   248
MixtralForCausalLM                   80
                                   ... 
Gemma2ForSequenceClassification       1
RwkvForCausalLM                       1
LlamaForRewardModelWithGating         1
Cohere2ForCausalLM                    1
SolarForCausalLM                      1
Name: count, Length: 64, dtype: int64

In [3]:
df

,fullname,Architecture,#Params (B),cohort
0,0-hero/Matter-0.2-7B-DPO,MistralForCausalLM,7.242,Q2-2024
1,01-ai/Yi-1.5-34B,LlamaForCausalLM,34.389,Q2-2024
2,01-ai/Yi-1.5-34B-32K,LlamaForCausalLM,34.389,Q2-2024
3,01-ai/Yi-1.5-34B-Chat,LlamaForCausalLM,34.389,Q2-2024
4,01-ai/Yi-1.5-34B-Chat-16K,LlamaForCausalLM,34.389,Q2-2024
...,...,...,...,...
4571,zelk12/recoilme-gemma-2-Ifable-9B-v0.1,Gemma2ForCausalLM,10.159,Q4-2024
4572,zelk12/recoilme-gemma-2-psy10k-mental_healt-9B...,Gemma2ForCausalLM,10.159,Q4-2024
4573,zetasepic/Qwen2.5-32B-Instruct-abliterated-v2,Qwen2ForCausalLM,32.764,Q4-2024
4574,zetasepic/Qwen2.5-72B-Instruct-abliterated,Qwen2ForCausalLM,72.706,Q4-2024


In [21]:
print(df.dtypes)

fullname         object
Architecture     object
#Params (B)     float64
cohort           object
dtype: object


In [23]:

def cohort_sort_key(cohort):
    if cohort == 'Unknown': return (9999, 9)
    q, year = cohort.split('-')
    return (int(year), int(q[1]))


# ── size bins ─────────────────────────────────────────────────────────────────
def size_bin(p):
    if pd.isna(p):   return "unknown"
    if p < 7:        return "<7B"
    if p <= 70:      return "7-70B"
    return ">70B"


df["size_bin"] = df["#Params (B)"].apply(size_bin)

cohorts = sorted(df["cohort"].dropna().unique(), key=cohort_sort_key)
ref_cohort = cohorts[0]
ref_params = df[df["cohort"] == ref_cohort]["#Params (B)"].dropna()

# ── Table 1 ───────────────────────────────────────────────────────────────────
rows = []
for cohort in cohorts:
    sub = df[df["cohort"] == cohort]
    params = sub["#Params (B)"].dropna()
    n = len(sub)
    n_known = len(params)

    pct_small = (sub["size_bin"] == "<7B").sum() / n * 100
    pct_mid = (sub["size_bin"] == "7-70B").sum() / n * 100
    pct_large = (sub["size_bin"] == ">70B").sum() / n * 100

    if cohort == ref_cohort:
        ks_stat, ks_p = np.nan, np.nan
    else:
        ks_stat, ks_p = ks_2samp(ref_params, params)

    rows.append({
        "cohort": cohort,
        "N": n,
        "mean_params": params.mean().round(1),
        "sd_params": params.std().round(1),
        "pct_<7B": round(pct_small, 1),
        "pct_7-70B": round(pct_mid, 1),
        "pct_>70B": round(pct_large, 1),
        "KS_stat": round(ks_stat, 3) if not np.isnan(ks_stat) else "ref",
        "p_value": round(ks_p, 4) if not np.isnan(ks_p) else "ref",
    })

table1 = pd.DataFrame(rows)
table1 = table1.sort_values("cohort", key=lambda s: s.map(cohort_sort_key))

print("=== Table 1: Cohort descriptive statistics ===")
print(table1.to_string(index=False))

=== Table 1: Cohort descriptive statistics ===
 cohort   N  mean_params  sd_params  pct_<7B  pct_7-70B  pct_>70B KS_stat p_value
Q2-2023  10         14.2       18.5     30.0       70.0       0.0     ref     ref
Q3-2023  28         19.8       23.8     17.9       82.1       0.0   0.179  0.9409
Q4-2023  37         13.3       14.5     40.5       59.5       0.0   0.178  0.9164
Q1-2024  29         24.3       23.9     17.2       72.4      10.3   0.328  0.3088
Q2-2024 247         18.6       23.1      7.3       80.2      12.6   0.427  0.0416
Q3-2024 288         12.0       16.4     18.1       76.0       5.9   0.389  0.0796
Q4-2024 415          9.4       14.2     33.3       62.2       4.6   0.433  0.0358
Q1-2025 538          8.8       10.2     29.2       68.6       2.2   0.323  0.2057
Unknown 257          9.9       10.4     21.0       77.4       1.6   0.333  0.1883


In [24]:
# ── Architecture KS analysis ──────────────────────────────────────────────────
# encode architecture as numeric frequency rank for KS test
arch_rank = df["Architecture"].value_counts().rank(ascending=False)
df["arch_rank"] = df["Architecture"].map(arch_rank)

ref_arch = df[df["cohort"] == ref_cohort]["arch_rank"].dropna()

rows_arch = []
for cohort in cohorts:
    sub      = df[df["cohort"] == cohort]["arch_rank"].dropna()
    n        = len(df[df["cohort"] == cohort])

    if cohort == ref_cohort:
        ks_stat, ks_p = np.nan, np.nan
    else:
        ks_stat, ks_p = ks_2samp(ref_arch, sub)

    # top 3 architectures in this cohort
    top3 = (df[df["cohort"] == cohort]["Architecture"]
            .value_counts()
            .head(3)
            .index.tolist())

    rows_arch.append({
        "cohort":   cohort,
        "N":        n,
        "top_arch": ", ".join(top3),
        "KS_stat":  round(ks_stat, 3) if not np.isnan(ks_stat) else "ref",
        "p_value":  round(ks_p, 4)    if not np.isnan(ks_p)    else "ref",
    })

table_arch = pd.DataFrame(rows_arch)
print("=== Architecture distribution KS analysis ===")
print(table_arch.to_string(index=False))

=== Architecture distribution KS analysis ===
 cohort   N         top_arch KS_stat p_value
Q2-2023  10 LlamaForCausalLM     ref     ref
Q3-2023  28 LlamaForCausalLM     0.0     1.0
Q4-2023  37 LlamaForCausalLM     0.0     1.0
Q1-2024  29 LlamaForCausalLM     0.0     1.0
Q2-2024 247 LlamaForCausalLM     0.0     1.0
Q3-2024 288 LlamaForCausalLM     0.0     1.0
Q4-2024 415 LlamaForCausalLM     0.0     1.0
Q1-2025 538 LlamaForCausalLM     0.0     1.0
Unknown 257 LlamaForCausalLM     0.0     1.0


In [25]:
# ── Table 2: Architecture descriptive statistics ──────────────────────────────
top_archs = df["Architecture"].value_counts().head(10).index
df_arch = df[df["Architecture"].isin(top_archs)]

ref_arch_name = df["Architecture"].value_counts().index[0]  # most common as reference
ref_params_arch = df[df["Architecture"] == ref_arch_name]["#Params (B)"].dropna()

rows_arch = []
for arch in top_archs:
    sub    = df[df["Architecture"] == arch]
    params = sub["#Params (B)"].dropna()
    n      = len(sub)

    pct_small = (sub["size_bin"] == "<7B").sum()  / n * 100
    pct_mid   = (sub["size_bin"] == "7-70B").sum() / n * 100
    pct_large = (sub["size_bin"] == ">70B").sum()  / n * 100

    if arch == ref_arch_name:
        ks_stat, ks_p = np.nan, np.nan
    else:
        ks_stat, ks_p = ks_2samp(ref_params_arch, params)

    rows_arch.append({
        "architecture": arch,
        "N":            n,
        "mean_params":  params.mean().round(1),
        "sd_params":    params.std().round(1),
        "pct_<7B":      round(pct_small, 1),
        "pct_7-70B":    round(pct_mid, 1),
        "pct_>70B":     round(pct_large, 1),
        "KS_stat":      round(ks_stat, 3) if not np.isnan(ks_stat) else "ref",
        "p_value":      round(ks_p, 4)    if not np.isnan(ks_p)    else "ref",
    })

table2 = pd.DataFrame(rows_arch)
table2 = table2.sort_values("N", ascending=False)

print("=== Table 2: Architecture descriptive statistics ===")
print(table2.to_string(index=False))

=== Table 2: Architecture descriptive statistics ===
    architecture    N  mean_params  sd_params  pct_<7B  pct_7-70B  pct_>70B KS_stat p_value
LlamaForCausalLM 1849         11.4       15.5     24.2       71.2       4.7     ref     ref
